# PhonePe Transaction Dispute Analyzer
## Python Developer Handover Challenge

**Simulation type:** Python Developer codebase handover challenge  
**Company context:** PhonePe support operations case study  
**Runtime:** Google Colab  
**Core tools:** Python, Pandas, ipywidgets, file handling  

A Product Manager has shared the PRD for an internal transaction dispute tool. A previous developer started the work and resigned before completing it. Only the basic transaction loading feature is stable. Your task is to understand the PRD, debug the existing notebook, complete the missing features, build the Colab interface, export final reports, and maintain traceability.


## Required Final Submission

Submit **one completed Colab notebook only**.

Everything required for evaluation must be visible inside this same notebook:

- Fixed code
- Visible output after every major section
- Debug fix log
- AI prompt usage log
- PRD completion mapping
- Assumption and limitation log
- Final reports preview
- Final product walkthrough
- Evaluation-ready explanation

The notebook may generate CSV files as product outputs, but those files are **not separate required submissions**.


# Meeting Handover Summary

**Product Manager:** The support operations team needs a tool to identify failed transactions, pending refunds, repeated complaints, dispute priority, and recommended actions.

**Engineering Manager:** The previous developer resigned. Only basic transaction loading works. The remaining code contains broken file loading, incorrect validation, buggy cleaning logic, wrong SLA calculations, incomplete priority rules, missing exports, and an unfinished Colab GUI.

**You:** You are the Python Developer now responsible for completing the product as per the PRD.

# Supporting Documents to Read

Before solving the notebook, read the two supporting documents in the `student_release` folder:

1. **[Meeting_Transcript_PhonePe_Transaction_Dispute_Analyzer ](https://docs.google.com/document/d/1PNt3DllUsqzPA2bGbc4nhHWX2-IezP_1/edit?usp=sharing&ouid=102899066126360955769&rtpof=true&sd=true)**— explains the workplace handover meeting.
2. **[PRD_PhonePe_Transaction_Dispute_Analyzer](https://docs.google.com/document/d/1_o9GQzUUkAhFjXjN5o5NtzjIbqXarCJe/edit?usp=sharing&ouid=102899066126360955769&rtpof=true&sd=true)** — defines the product requirements, business rules, reports, acceptance criteria, and constraints.

Your implementation should map back to the PRD. Keep the mapping and explanation inside this same notebook.

# Dataset Upload Instructions

Upload `phonepe_transaction_dispute_dataset.zip` when prompted. The code below will extract it automatically.

Expected files:

- `transactions.csv` — 5,000 rows
- `refunds.xlsx` — 1,300 rows
- `customer_complaints.json` — 1,500 rows
- `support_tickets.csv` — 1,500 rows
- `customers.csv` — 1,200 rows
- `merchants.csv` — 1,000 rows
- `status_notes.txt` — 100 notes

In [1]:
# ============================================================
# Upload and extract dataset
# ============================================================

import os, zipfile, glob, json, warnings
from pathlib import Path

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

DATASET_FOLDER_NAME = "phonepe_transaction_dispute_dataset"
DATASET_ZIP_NAME = "phonepe_transaction_dispute_dataset.zip"

def prepare_dataset():
    """Find or extract the dataset folder in the current Colab/session directory."""
    if os.path.isdir(DATASET_FOLDER_NAME):
        print(f"Dataset folder found: {DATASET_FOLDER_NAME}")
        return DATASET_FOLDER_NAME

    if os.path.exists(DATASET_ZIP_NAME):
        print(f"Extracting existing {DATASET_ZIP_NAME}...")
        os.makedirs(DATASET_FOLDER_NAME, exist_ok=True)
        with zipfile.ZipFile(DATASET_ZIP_NAME, "r") as z:
            z.extractall(DATASET_FOLDER_NAME)
        print(f"Dataset extracted to: {DATASET_FOLDER_NAME}")
        return DATASET_FOLDER_NAME

    if IN_COLAB:
        print("Upload phonepe_transaction_dispute_dataset.zip when prompted.")
        uploaded = files.upload()
        zip_candidates = [name for name in uploaded.keys() if name.endswith(".zip")]
        if not zip_candidates:
            raise FileNotFoundError("No ZIP file uploaded. Please upload phonepe_transaction_dispute_dataset.zip")
        os.makedirs(DATASET_FOLDER_NAME, exist_ok=True)
        with zipfile.ZipFile(zip_candidates[0], "r") as z:
            z.extractall(DATASET_FOLDER_NAME)
        print(f"Dataset extracted to: {DATASET_FOLDER_NAME}")
        return DATASET_FOLDER_NAME

    raise FileNotFoundError("Dataset ZIP/folder not found. Place phonepe_transaction_dispute_dataset.zip in the runtime and rerun.")

DATASET_PATH = prepare_dataset()
print("DATASET_PATH =", DATASET_PATH)
print("Files available:", sorted(os.listdir(DATASET_PATH)))

Upload phonepe_transaction_dispute_dataset.zip when prompted.


Saving phonepe_transaction_dispute_dataset (1).zip to phonepe_transaction_dispute_dataset (1).zip
Dataset extracted to: phonepe_transaction_dispute_dataset
DATASET_PATH = phonepe_transaction_dispute_dataset
Files available: ['phonepe_transaction_dispute_dataset']


In [2]:
# ============================================================
# Section 1: Imports and configuration
# Status: Working
# ============================================================

import pandas as pd
import numpy as np
from datetime import datetime

ANALYSIS_DATE = pd.Timestamp("2026-06-17")
OUTPUT_PATH = Path("phonepe_dispute_outputs")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Libraries imported successfully.")
print("Analysis date:", ANALYSIS_DATE.date())

Libraries imported successfully.
Analysis date: 2026-06-17


### Expected Output — Section 1: Imports and Configuration

**Datasets to use:**

- No dataset required in this section.

**How to approach:**

- Import only libraries needed for this Colab project.
- Keep configuration values such as `ANALYSIS_DATE` and `OUTPUT_PATH` in one place.
- Use the fixed analysis date from the PRD so refund and ticket SLA calculations are reproducible.

**Expected output format:**

Show clear confirmation output:

| Configuration | Expected Value |
|---|---|
| Libraries | Imported successfully |
| `ANALYSIS_DATE` | `2026-06-17` |
| `OUTPUT_PATH` | `phonepe_dispute_outputs` |

**Hint:** If you later need constants such as SLA thresholds or valid priority labels, define them here instead of hard-coding them across many cells.


In [4]:
import os

# ============================================================
# Section 2: Working feature - Load transactions.csv
# Status: Working
# ============================================================

transactions_path = os.path.join(DATASET_PATH, DATASET_FOLDER_NAME, "transactions.csv")
transactions_df = pd.read_csv(transactions_path)

print("transactions.csv loaded successfully")
print("Shape:", transactions_df.shape)
transactions_df.head()

transactions.csv loaded successfully
Shape: (5000, 14)


,transaction_id,customer_id,merchant_id,transaction_date,transaction_time,amount,payment_mode,transaction_status,failure_reason,bank_name,city,device_type,app_version,merchant_category
0,TXN0002999,CUST000997,MERCH0410,2026-05-07,15:28:08,221.28,UPI,Success,NaN,Punjab National Bank,Gurugram,Android,24.1.0,Food
1,TXN0004835,CUST000270,MERCH0097,2026-05-10,07:59:23,13926.17,card,Success,NaN,Kotak Mahindra Bank,Pune,Android,24.1.0,Healthcare
2,TXN0001281,CUST000988,MERCH0956,2026-05-18,03:31:58,3112.07,Bank Transfer,Failed,UPI Limit Exceeded,SBI,Mumbai,Android,24.1.1,Grocery
3,TXN0003263,CUST000777,MERCH0314,2026-05-13,12:07:16,8276.33,UPI,Pending,NPCI Processing Delay,Canara Bank,Kolkata,Android,23.9.1,Utility
4,TXN0003202,CUST000867,MERCH0849,2026-06-14,17:43:52,1240.12,net banking,Success,NaN,Kotak Mahindra Bank,Ahmedabad,Android,24.0.2,Grocery


### Expected Output — Section 2: Load `transactions.csv`

**Datasets to use:**

- `transactions.csv`

**How to approach:**

- Confirm the dataset folder path is correct.
- Use `pd.read_csv()` to load the transaction file.
- Do not clean the data in this section yet; only verify that the raw file loads.

**Expected output format:**

- A success message such as `transactions.csv loaded successfully`.
- Shape close to **`(5000, 14)`**.
- A visible `head()` preview with columns like:
  - `transaction_id`
  - `customer_id`
  - `merchant_id`
  - `transaction_date`
  - `amount`
  - `payment_mode`
  - `transaction_status`

**Student note:** This is the only feature the previous developer completed properly. Use this section as your reference style for clear output messages.


In [5]:
# ============================================================
# Section 3: Working feature - Basic transaction summary
# Status: Working but too basic
# ============================================================

print("Total rows:", len(transactions_df))
print("Total columns:", len(transactions_df.columns))
print("Available columns:")
print(list(transactions_df.columns))

transactions_df["transaction_status"].value_counts(dropna=False)

Total rows: 5000
Total columns: 14
Available columns:
['transaction_id', 'customer_id', 'merchant_id', 'transaction_date', 'transaction_time', 'amount', 'payment_mode', 'transaction_status', 'failure_reason', 'bank_name', 'city', 'device_type', 'app_version', 'merchant_category']


,count
transaction_status,
Success,3177
Failed,804
Pending,432
Reversed,263
successful,59
success,54
SUCCESS,53
completed,52
REVERSED,13


### Expected Output — Section 3: Basic Transaction Summary

**Datasets to use:**

- `transactions.csv`

**How to approach:**

- Display total rows, total columns, and the raw column list.
- Check raw value counts for `transaction_status`.
- Observe messy status values before cleaning. Do not fix them here yet.

**Expected output format:**

Create visible output showing:

| Output Item | Expected Evidence |
|---|---|
| Total rows | Around `5000` |
| Total columns | Around `14` |
| Column list | All transaction columns visible |
| Raw status counts | Includes clean and messy values such as success/failed/pending variants |

**Hint:** Later sections should standardize these messy status values into business-ready categories such as `Success`, `Failed`, `Pending`, and `Reversed`.


---
# Developer Work Starts Here

The following code was left by the previous developer. Some sections are broken, some are incomplete, and some produce incorrect business results even if they run.

Rules:

1. Do not simply delete the notebook and start from scratch.
2. Debug and improve the existing codebase section by section.
3. Maintain a debug log.
4. Map every completed feature to the PRD completion checklist.
5. Use AI if needed, but verify every answer and maintain an AI prompt log.

In [6]:
# ============================================================
# Section 4: Debug log setup
# Status: Incomplete
# ============================================================

# TODO: Convert this into a proper debug log workflow.
# Required columns:
# issue_id, code_section, issue_type, issue_description,
# root_cause, fix_summary, tested_status, remarks

debug_log = []

def add_debug_log(issue_id, section, issue_type, description):
    # BUG: This function does not capture all required fields.
    debug_log.append([issue_id, section, issue_type, description])

add_debug_log("BUG-001", "Setup", "handover", "Initial debug log created")
debug_log

[['BUG-001', 'Setup', 'handover', 'Initial debug log created']]

### Expected Output — Section 4: Debug Log Setup

**Datasets to use:**

- No dataset required in this section.

**How to approach:**

- Create a reusable debug log structure before fixing the project.
- Every time you fix a meaningful bug, add one row to the log.
- Keep this log visible inside the notebook and export it at the end.

**Expected output format:**

Create a visible table named `debug_fix_log_df` with these columns:

| Column | What to write |
|---|---|
| `issue_id` | Example: `BUG-001` |
| `code_section` | Example: `Section 5 - DataLoader` |
| `issue_type` | Syntax, runtime, logic, data, OOP, export, GUI |
| `issue_description` | What was broken |
| `root_cause` | Why it was broken |
| `fix_summary` | What you changed |
| `tested_status` | Passed / Failed |
| `remarks` | Any extra note |

**Minimum requirement:** At least **10 meaningful fixes** must be documented before final submission.

**Hint:** Do not fill this only at the end from memory. Update it section by section while you debug.


In [8]:
# ============================================================
# Section 5: Previous developer's generic data loader
# Status: Broken
# ============================================================

class DataLoader:
    def __init__(self, folder_path):
        self.folder_path = folder_path

    def load_csv(self, file_name):
        return pd.read_csv(os.path.join(self.folder_path, file_name))

    def load_excel(self, file_name):
        # BUG: Previous developer incorrectly used read_csv for Excel files.
        return pd.read_excel(os.path.join(self.folder_path, file_name))

    def load_json(self, file_name):
        # BUG: This assumes JSON is line-delimited, but the dataset is a JSON list.
        return pd.read_json(os.path.join(self.folder_path, file_name))

    def load_text(self, file_name):
        # BUG: Missing encoding handling and no clean line parsing.
        with open(os.path.join(self.folder_path, file_name), 'r', encoding='utf-8') as f:
            return [line.strip() for line in f.readlines() if line.strip()]

loader = DataLoader(os.path.join(DATASET_PATH, DATASET_FOLDER_NAME))

# TODO: Fix the loader so all files load correctly.
refunds_df = loader.load_excel("refunds.xlsx")
complaints_df = loader.load_json("customer_complaints.json")
tickets_df = loader.load_csv("support_tickets.csv")
customers_df = loader.load_csv("customers.csv")
merchants_df = loader.load_csv("merchants.csv")
status_notes = loader.load_text("status_notes.txt")


### Expected Output — Section 5: Multi-File Data Loader

**Datasets to use:**

- `transactions.csv`
- `refunds.xlsx`
- `customer_complaints.json`
- `support_tickets.csv`
- `customers.csv`
- `merchants.csv`
- `status_notes.txt`

**How to approach:**

- Fix the `DataLoader` class so it can load CSV, Excel, JSON list, and text files.
- Excel files should use `pd.read_excel()`.
- The complaint JSON is a JSON list, not line-delimited JSON.
- Text notes should load with safe encoding and return clean lines.
- Create a loading summary table after all files are loaded.

**Expected output format:**

Display a table similar to this:

| file_name | expected_rows | actual_rows | columns_or_items | status |
|---|---:|---:|---:|---|
| transactions.csv | 5000 | 5000 | 14 | Loaded |
| refunds.xlsx | 1300 | 1300 | 8 | Loaded |
| customer_complaints.json | 1500 | 1500 | 8 | Loaded |
| support_tickets.csv | 1500 | 1500 | 8 | Loaded |
| customers.csv | 1200 | 1200 | 6 | Loaded |
| merchants.csv | 1000 | 1000 | 6 | Loaded |
| status_notes.txt | 100 | 100 | text lines | Loaded |

**Hint:** Make the loader reusable. Do not write seven unrelated loading statements if a class or helper function can handle this more cleanly.


In [9]:
# ============================================================
# Section 6: Data validation engine
# Status: Broken / Incomplete
# ============================================================

required_columns = {
    "transactions": ["txn_id", "customer_id", "amount", "status"],  # BUG: wrong column names
    "refunds": ["refund_id", "transaction_id", "refund_status"],
    # TODO: Add complaints, tickets, customers, merchants validations
}

class DataValidator:
    def __init__(self, required_columns):
        self.required_columns = required_columns

    def validate(self, dataset_name, df):
        missing = []
        for col in self.required_columns[dataset_name]:
            if col not in df.columns:
                missing.append(col)
        return missing

validator = DataValidator(required_columns)

# BUG: This will either fail or give misleading validation results.
validator.validate("transactions", transactions_df)

['txn_id', 'status']

### Expected Output — Section 6: Data Validation Engine

**Datasets to use:**

- All loaded datasets from Section 5.

**How to approach:**

- Create a dictionary of required columns for each dataset.
- Validate required columns, extra columns, duplicate IDs, nulls in critical columns, and orphan records.
- Critical fields include IDs such as `transaction_id`, `customer_id`, `merchant_id`, `refund_id`, `complaint_id`, and `ticket_id`.
- Keep validation warnings visible instead of silently ignoring issues.

**Expected output format:**

Create a visible table named `data_validation_report` with columns like:

| dataset | check_type | column_or_key | issue_count | severity | status | message |
|---|---|---|---:|---|---|---|
| transactions | Required Columns | transaction_id | 0 | Critical | Passed | Required column present |
| refunds | Orphan Transaction IDs | transaction_id | some number | Warning | Review | Some refund records may not match transactions |

**Hint:** Separate **critical errors** from **warnings**. Critical errors should stop the pipeline; warnings should be logged and reviewed.


In [10]:
# ============================================================
# Section 7: Data cleaning utilities
# Status: Buggy
# ============================================================

status_map = {
    "success": "Success",
    "failed": "Failed",
    "pending": "Pending",
    "reversed": "Reversed"
}

payment_mode_map = {
    "upi": "UPI",
    "wallet": "Wallet",
    "card": "Card",
    "bank": "Bank Transfer"
}

def clean_status_column(df, column):
    # BUG: Does not handle spaces, uppercase, alternate values like failure/declined/processing.
    df[column] = df[column].map(status_map)
    return df

def clean_amount_column(df, column):
    # BUG: Does not handle currency symbols, commas, blanks, negative values, or invalid strings.
    df[column] = df[column].astype(float)
    return df

def clean_date_column(df, column):
    # BUG: Does not handle invalid date values gracefully.
    df[column] = pd.to_datetime(df[column])
    return df

# TODO: Apply correct cleaning across all datasets.
clean_transactions_df = clean_status_column(transactions_df.copy(), "transaction_status")
clean_transactions_df.head()

,transaction_id,customer_id,merchant_id,transaction_date,transaction_time,amount,payment_mode,transaction_status,failure_reason,bank_name,city,device_type,app_version,merchant_category
0,TXN0002999,CUST000997,MERCH0410,2026-05-07,15:28:08,221.28,UPI,NaN,NaN,Punjab National Bank,Gurugram,Android,24.1.0,Food
1,TXN0004835,CUST000270,MERCH0097,2026-05-10,07:59:23,13926.17,card,NaN,NaN,Kotak Mahindra Bank,Pune,Android,24.1.0,Healthcare
2,TXN0001281,CUST000988,MERCH0956,2026-05-18,03:31:58,3112.07,Bank Transfer,NaN,UPI Limit Exceeded,SBI,Mumbai,Android,24.1.1,Grocery
3,TXN0003263,CUST000777,MERCH0314,2026-05-13,12:07:16,8276.33,UPI,NaN,NPCI Processing Delay,Canara Bank,Kolkata,Android,23.9.1,Utility
4,TXN0003202,CUST000867,MERCH0849,2026-06-14,17:43:52,1240.12,net banking,NaN,NaN,Kotak Mahindra Bank,Ahmedabad,Android,24.0.2,Grocery


### Expected Output — Section 7: Data Cleaning and Standardization

**Datasets to use:**

- `transactions.csv`
- `refunds.xlsx`
- `customer_complaints.json`
- `support_tickets.csv`
- `customers.csv`
- `merchants.csv`

**How to approach:**

- Standardize transaction statuses, refund statuses, complaint statuses, ticket statuses, payment modes, dates, text fields, and amounts.
- Convert date columns using `pd.to_datetime(..., errors='coerce')`.
- Convert amount columns using numeric conversion with safe error handling.
- Fill missing business fields using PRD rules.
- Preserve raw IDs; do not create random IDs during cleaning.

**Expected output format:**

Show before/after evidence, for example:

| Cleaning Area | Before Evidence | After Evidence |
|---|---|---|
| Transaction status | Raw messy value counts | Standardized value counts |
| Payment mode | Raw messy payment modes | `UPI`, `Wallet`, `Card`, `Bank Transfer` |
| Date columns | Object/string types | Datetime types |
| Amount columns | Mixed/string values | Numeric dtype |

**Hint:** Do not over-clean by deleting rows aggressively. Most issues should be standardized, flagged, or filled according to business rules.


In [11]:
# ============================================================
# Section 8: Refund delay analysis
# Status: Wrong business logic
# ============================================================

def calculate_refund_delay(row):
    # BUG: Uses completed date even for pending refunds.
    # BUG: Does not use fixed ANALYSIS_DATE.
    return (pd.to_datetime(row["refund_completed_date"]) - pd.to_datetime(row["refund_initiated_date"])).days

def classify_refund_issue(row):
    # TODO: Implement Refund Completed On Time, Refund Delay Warning,
    # Refund SLA Breach, Refund Failed, Refund Record Missing,
    # Partial Refund, Refund Amount Mismatch.
    if row["refund_status"] == "Pending":
        return "Pending"
    return "OK"

refunds_df["refund_delay_days"] = refunds_df.apply(calculate_refund_delay, axis=1)
refunds_df["refund_issue_tag"] = refunds_df.apply(classify_refund_issue, axis=1)
refunds_df.head()

,refund_id,transaction_id,refund_status,refund_amount,refund_initiated_date,refund_completed_date,refund_reason,refund_channel,refund_delay_days,refund_issue_tag
0,REF001197,TXN0002159,Failed,739.7,2026-06-06,NaN,Duplicate Debit,Wallet,NaN,OK
1,REF000547,TXN0004630,Not Applicable,2459,2026-06-02,NaN,Bank Reversal,Bank Transfer,NaN,OK
2,REF000007,TXN0001558,Pending,1451.18,2026-05-11,NaN,Technical Failure,Wallet,NaN,Pending
3,REF000081,TXN0003820,Pending,3606.99,2026-05-25,NaN,Customer Dispute,Bank Transfer,NaN,Pending
4,REF000613,TXN0004256,Initiated,27616.25,2026-05-29,NaN,Customer Dispute,Bank Transfer,NaN,OK


### Expected Output — Section 8: Transaction Failure Analysis

**Datasets to use:**

- Cleaned `transactions_df`

**How to approach:**

- Use standardized `transaction_status` values.
- Calculate total transactions, status counts, success rate, failure rate, pending count, reversed count, and failed amount.
- Handle division by zero safely.

**Expected output format:**

Create a visible `transaction_health_summary` table or dictionary with:

| Metric | Expected Type |
|---|---|
| total_transactions | integer |
| successful_transactions | integer |
| failed_transactions | integer |
| pending_transactions | integer |
| reversed_transactions | integer |
| success_rate | percentage/float |
| failure_rate | percentage/float |
| total_failed_amount | numeric |

**Hint:** This section should not depend on refunds or complaints yet. Keep it focused only on transaction health.


In [12]:
# ============================================================
# Section 9: Complaint linking engine
# Status: Broken merge logic
# ============================================================

# BUG: Direct merge can duplicate transaction rows when multiple complaints exist.
# TODO: Aggregate complaints per transaction before merging.
complaint_linked_df = transactions_df.merge(complaints_df, on="transaction_id", how="left")
complaint_linked_df.shape

(5355, 21)

### Expected Output — Section 9: Refund Delay and SLA Analysis

**Datasets to use:**

- Cleaned `transactions_df`
- Cleaned `refunds_df`

**How to approach:**

- Link refunds to transactions using `transaction_id`.
- Use fixed `ANALYSIS_DATE = 2026-06-17` for reproducible delay calculation.
- Calculate `refund_delay_days` using completed date when available, otherwise analysis date.
- Classify refund issue tags based on PRD rules.

**Expected output format:**

Create refund-related columns such as:

| Column | Meaning |
|---|---|
| `refund_status` | Standardized refund status |
| `refund_delay_days` | Number of days between initiation and completion/analysis date |
| `refund_issue_tag` | `Refund Completed On Time`, `Refund Delay Warning`, `Refund SLA Breach`, `Refund Failed`, `Refund Record Missing`, `Partial Refund`, `Refund Amount Mismatch` |

Also show a value-count summary for `refund_issue_tag`.

**Hint:** Failed transactions with no refund record are important. Do not lose them during merge; use a left merge from transactions to refunds.


In [13]:
# ============================================================
# Section 10: Support ticket mapping
# Status: Incomplete
# ============================================================

def classify_ticket_severity(row):
    # BUG: Does not consider ticket age, escalation flag, or slow resolution properly.
    if row["ticket_status"] == "Escalated":
        return "High"
    return "Normal"

# TODO: Aggregate multiple tickets per transaction.
# TODO: Calculate ticket age against ANALYSIS_DATE.
# TODO: Implement Ticket Delay Warning, Ticket SLA Breach, Escalated Case, Slow Resolution.

tickets_df["ticket_severity"] = tickets_df.apply(classify_ticket_severity, axis=1)
tickets_df.head()

,ticket_id,transaction_id,customer_id,ticket_created_date,ticket_status,assigned_team,resolution_time_hours,escalation_flag,ticket_severity
0,TKT000640,TXN0002830,CUST000470,2026-06-11,Resolved,Refund Ops,123.0,No,Normal
1,TKT000150,TXN0001558,CUST000125,2026-05-13,Pending,Tech Support,34.0,No,Normal
2,TKT000411,TXN0004596,CUST000993,2026-05-31,Open,Merchant Ops,NaN,No,Normal
3,TKT000292,TXN0001348,CUST000019,2026-06-03,Open,Bank Ops,NaN,No,Normal
4,TKT000151,TXN0001606,CUST000676,2026-06-13,Resolved,Refund Ops,43.0,No,Normal


### Expected Output — Section 10: Complaint Linking Engine

**Datasets to use:**

- Cleaned `transactions_df`
- Cleaned `customer_complaints.json`
- Cleaned `customers_df` if needed for customer-level repeat history

**How to approach:**

- Link complaints using `transaction_id` and verify `customer_id` consistency.
- Count complaints per transaction.
- Count complaints per customer.
- Identify reopened, angry, unresolved, repeated, and orphan complaints.
- Aggregate complaint details before merging into final transaction-level data.

**Expected output format:**

Create transaction-level columns such as:

| Column | Meaning |
|---|---|
| `complaint_count` | Number of complaints linked to the transaction |
| `customer_complaint_count` | Complaints raised by the customer across transactions |
| `complaint_status` | Latest or highest-priority complaint status |
| `complaint_type` | Main complaint category |
| `sentiment_tag` | Latest/highest-severity sentiment |
| `complaint_severity` | `Severe`, `Moderate`, `Low`, `None` |
| `repeated_complaint_flag` | Yes/No |

**Hint:** If multiple complaints exist for one transaction, group first, then merge. Direct merging can accidentally increase transaction rows.


In [14]:
# ============================================================
# Section 11: Merge master data
# Status: Risky
# ============================================================

# BUG: This assumes all previous steps succeeded and does not handle duplicated columns.
# BUG: Missing merchant category should be filled from merchant master.

merged_df = transactions_df.merge(customers_df, on="customer_id", how="left")
merged_df = merged_df.merge(merchants_df, on="merchant_id", how="left")
merged_df = merged_df.merge(refunds_df, on="transaction_id", how="left")

merged_df.head()

,transaction_id,customer_id,merchant_id,transaction_date,transaction_time,amount,payment_mode,transaction_status,failure_reason,bank_name,city,device_type,app_version,merchant_category_x,customer_name,customer_segment,account_age_days,total_transactions,preferred_payment_mode,merchant_name,merchant_category_y,merchant_city,merchant_rating,monthly_transaction_volume,refund_id,refund_status,refund_amount,refund_initiated_date,refund_completed_date,refund_reason,refund_channel,refund_delay_days,refund_issue_tag
0,TXN0002999,CUST000997,MERCH0410,2026-05-07,15:28:08,221.28,UPI,Success,NaN,Punjab National Bank,Gurugram,Android,24.1.0,Food,Arjun Pillai,New,105,3,UPI,Urban Food Point 410,Food,Pune,3.7,662,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,TXN0004835,CUST000270,MERCH0097,2026-05-10,07:59:23,13926.17,card,Success,NaN,Kotak Mahindra Bank,Pune,Android,24.1.0,Healthcare,Vihaan Singh,Regular,862,4,Card,Wellness Healthcare Kart 97,Healthcare,Pune,4.6,1334,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,TXN0001281,CUST000988,MERCH0956,2026-05-18,03:31:58,3112.07,Bank Transfer,Failed,UPI Limit Exceeded,SBI,Mumbai,Android,24.1.1,Grocery,Aadhya Jain,Premium,1455,16,UPI,Basket Grocery Online 956,Grocery,Mumbai,3.9,743,REF000477,Initiated,3112.07,2026-05-18,NaN,Technical Failure,Bank Transfer,NaN,OK
3,TXN0003263,CUST000777,MERCH0314,2026-05-13,12:07:16,8276.33,UPI,Pending,NPCI Processing Delay,Canara Bank,Kolkata,Android,23.9.1,Utility,Yash Malhotra,Regular,878,5,UPI,Civic Utility Plus 314,Utility,Kolkata,2.6,272,REF001151,Pending,4834.63,2026-05-16,NaN,Failed Payment,Bank Transfer,NaN,Pending
4,TXN0003202,CUST000867,MERCH0849,2026-06-14,17:43:52,1240.12,net banking,Success,NaN,Kotak Mahindra Bank,Ahmedabad,Android,24.0.2,Grocery,Kavya Sharma,Regular,339,5,Bank Transfer,QuickKart Grocery Services 849,Grocery,Ahmedabad,4.4,209,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Expected Output — Section 11: Support Ticket Mapping

**Datasets to use:**

- Cleaned `transactions_df`
- Cleaned `support_tickets.csv`
- Complaint summary from Section 10 if needed

**How to approach:**

- Link tickets using `transaction_id`.
- Calculate ticket age using `ANALYSIS_DATE` for open/pending/escalated tickets.
- Use `resolution_time_hours` for resolved/closed tickets.
- Group multiple tickets per transaction before merging.
- Create ticket severity based on PRD SLA rules.

**Expected output format:**

Create columns such as:

| Column | Meaning |
|---|---|
| `ticket_count` | Number of tickets linked to the transaction |
| `ticket_status` | Latest/highest-priority ticket status |
| `assigned_team` | Team handling the issue |
| `escalation_flag` | Yes/No |
| `ticket_age_hours` | Age for active tickets |
| `ticket_severity` | `Severe`, `Moderate`, `Low`, `None` |
| `ticket_issue_tag` | `Ticket SLA Breach`, `Ticket Delay Warning`, `Escalated Case`, `Slow Resolution`, etc. |

**Hint:** Ticket SLA logic should not depend only on `ticket_status`; escalation and resolution time also matter.


In [15]:
# ============================================================
# Section 12: Duplicate transaction detection
# Status: Placeholder
# ============================================================

def detect_duplicate_transaction(row):
    # TODO: Implement PRD duplicate detection logic.
    # Same customer, merchant, amount, date, close timestamp, and mixed success/failed/pending status.
    return "No"

merged_df["duplicate_suspected"] = merged_df.apply(detect_duplicate_transaction, axis=1)
merged_df["duplicate_suspected"].value_counts()

,count
duplicate_suspected,
No,5000


### Expected Output — Section 12: Duplicate Transaction Detection

**Datasets to use:**

- Cleaned `transactions_df`

**How to approach:**

- Compare transactions for the same `customer_id`, `merchant_id`, `amount`, and transaction date.
- Use transaction time difference to detect possible duplicate debits.
- Flag suspicious records; do not delete them.
- The PRD expects duplicate suspicion, not automatic removal.

**Expected output format:**

Create columns such as:

| Column | Meaning |
|---|---|
| `duplicate_suspected` | Yes/No |
| `duplicate_group_key` | Optional grouping key used for detection |
| `duplicate_reason` | Short explanation of why duplicate is suspected |

Show:

- Count of duplicate-suspected cases.
- A preview of 5–10 suspicious records.

**Hint:** Start simple with groupby on customer, merchant, amount, date. Then improve by considering transaction timestamps and status combinations.


In [18]:
# ============================================================
# Section 13: Customer impact score
# Status: Incorrect and unbounded
# ============================================================

def calculate_impact_score(row):
    # BUG: Score can exceed 100.
    # BUG: Does not handle missing values safely.
    score = 0
    # 'amount' column is already ensured to be numeric and NaNs filled before this apply
    score += row["amount"] / 100
    if row["customer_segment"] == "Premium":
        score += 30
    if row.get("complaint_count", 0) > 1:
        score += 50
    return score

# Ensure 'amount' column in merged_df is numeric before applying the function
# This line is crucial for fixing the TypeError if 'amount' is still a string.
merged_df["amount"] = pd.to_numeric(merged_df["amount"], errors='coerce').fillna(0)

merged_df["customer_impact_score"] = merged_df.apply(calculate_impact_score, axis=1)
merged_df[["transaction_id", "customer_impact_score"]].head()

,transaction_id,customer_impact_score
0,TXN0002999,2.2128
1,TXN0004835,139.2617
2,TXN0001281,61.1207
3,TXN0003263,82.7633
4,TXN0003202,12.4012


### Expected Output — Section 13: Final Feature Table / Master Merge

**Datasets to use:**

- Cleaned transactions
- Refund analysis output
- Complaint summary output
- Ticket summary output
- Customers master
- Merchants master
- Duplicate detection output

**How to approach:**

- Build one transaction-level master table.
- Preserve one row per `transaction_id` as much as possible.
- Use left joins from transactions to all feature tables.
- Fill missing feature columns with business-readable defaults such as `No Complaint`, `No Ticket`, `Not Available`, or `No`.

**Expected output format:**

Create a DataFrame such as `final_feature_df` or `merged_df` with one row per transaction and columns from all business areas:

| Area | Example Columns |
|---|---|
| Transaction | `transaction_id`, `customer_id`, `merchant_id`, `amount`, `payment_mode`, `transaction_status` |
| Refund | `refund_status`, `refund_delay_days`, `refund_issue_tag` |
| Complaint | `complaint_count`, `complaint_status`, `complaint_severity`, `sentiment_tag` |
| Ticket | `ticket_count`, `ticket_status`, `ticket_severity`, `escalation_flag` |
| Customer | `customer_segment`, `total_transactions`, `preferred_payment_mode` |
| Merchant | `merchant_name`, `merchant_category`, `merchant_city`, `merchant_rating` |
| Duplicate | `duplicate_suspected`, `duplicate_reason` |

**Hint:** After merging, check if row count has unexpectedly increased. If yes, you probably merged raw one-to-many complaint/ticket data instead of grouped summaries.


### Expected Output — Section 14: Dispute Priority Classification

**Datasets to use:**

- Final feature table from Section 13

**How to approach:**

- Implement priority rules from the PRD.
- Apply hierarchy in this exact order: **P0 > P1 > P2 > P3 > No Issue**.
- Generate both `dispute_priority` and `priority_reason`.
- Every transaction must receive exactly one priority label.

**Expected output format:**

Create columns:

| Column | Meaning |
|---|---|
| `dispute_priority` | `P0`, `P1`, `P2`, `P3`, or `No Issue` |
| `priority_reason` | Business-readable reason for the assigned priority |

Show:

- `value_counts()` for `dispute_priority`.
- Sample P0, P1, P2, P3, and No Issue rows.

**Hint:** Avoid a flat `if/elif` that misses important rules. Write helper functions for P0/P1/P2/P3 checks or keep the logic very readable.


In [20]:
# ============================================================
# Section 15: Recommended action generator
# Status: Too generic
# ============================================================

def generate_recommended_action(row):
    # TODO: Replace this with rule-based recommendations from PRD.
    if row["dispute_priority"] in ["P0", "P1"]:
        return "Escalate"
    return "Monitor"

merged_df["recommended_action"] = merged_df.apply(generate_recommended_action, axis=1)
merged_df[["transaction_id", "dispute_priority", "recommended_action"]].head()

,transaction_id,dispute_priority,recommended_action
0,TXN0002999,No Issue,Monitor
1,TXN0004835,No Issue,Monitor
2,TXN0001281,P3,Monitor
3,TXN0003263,P3,Monitor
4,TXN0003202,No Issue,Monitor


### Expected Output — Section 15: Recommended Action Generator

**Datasets to use:**

- Final table with `dispute_priority`, refund issue, complaint severity, ticket severity, customer segment, duplicate flag, and merchant details.

**How to approach:**

- Convert business rules into readable action messages.
- Recommended actions should help a support or operations user decide the next step.
- Do not return only generic values like `Escalate` or `Monitor`.

**Expected output format:**

Create a column:

| Column | Meaning |
|---|---|
| `recommended_action` | Clear next action for operations team |

Examples of acceptable action style:

- `Escalate to Refund Operations and review refund SLA breach immediately.`
- `Escalate to Bank Operations for duplicate debit investigation.`
- `Arrange immediate callback for Premium customer and update ticket notes.`
- `No action required; monitor through normal transaction health checks.`

**Hint:** Recommended action should be based on the strongest issue in the case, not just the priority label.


In [21]:
# ============================================================
# Section 16: AI-ready support prompt generator
# Status: Weak / incomplete
# ============================================================

def generate_ai_prompt(row):
    # BUG: Prompt is not structured enough and misses refund/ticket/priority context.
    # TODO: Generate prompts only for P0, P1, and P2.
    return f"Write reply for transaction {row['transaction_id']}"

merged_df["ai_support_prompt"] = merged_df.apply(generate_ai_prompt, axis=1)
merged_df[["transaction_id", "ai_support_prompt"]].head()

,transaction_id,ai_support_prompt
0,TXN0002999,Write reply for transaction TXN0002999
1,TXN0004835,Write reply for transaction TXN0004835
2,TXN0001281,Write reply for transaction TXN0001281
3,TXN0003263,Write reply for transaction TXN0003263
4,TXN0003202,Write reply for transaction TXN0003202


### Expected Output — Section 16: AI-Ready Support Prompt Generator

**Datasets to use:**

- Final report table with priority, refund, complaint, ticket, and recommended action columns.

**How to approach:**

- Generate prompts only for `P0`, `P1`, and `P2` cases.
- The prompt should help a support executive draft a customer response.
- Do not include unnecessary personal information such as full customer name if not required.
- Keep prompts professional and safe: no false promises, no confidential details, no unsupported claims.

**Expected output format:**

Create a column:

| Column | Meaning |
|---|---|
| `ai_support_prompt` | Structured prompt for an AI assistant |

Also create and display an `ai_prompt_log_df` in the deliverables section with columns:

| Column | Meaning |
|---|---|
| `prompt_id` | Example: `AI-001` |
| `section_used` | Example: `Section 16` |
| `prompt_goal` | What you asked AI to help with |
| `prompt_used` | Your actual prompt |
| `ai_output_summary` | What AI suggested |
| `accepted_or_modified` | Accepted / Modified / Rejected |
| `verification_notes` | How you verified correctness |

**Hint:** The output prompt is part of the product. The AI prompt usage log is part of your learning/process documentation. Both must be visible in the notebook.


In [22]:
# ============================================================
# Section 17: Colab transaction search GUI
# Status: Broken
# ============================================================

# TODO: Fix this interface using ipywidgets.
# Required: transaction_id input, search button, readable output, invalid ID handling.

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    transaction_input = widgets.Text(description="Txn ID:")
    search_button = widgets.Button(description="Search")
    output_area = widgets.Output()

    def search_transaction(button):
        with output_area:
            clear_output()
            txn_id = transaction_input.value
            # BUG: final_report is not defined.
            result = final_report[final_report["transaction_id"] == txn_id]
            display(result)

    search_button.on_click(search_transaction)
    display(transaction_input, search_button, output_area)
except Exception as e:
    print("GUI section needs fixing:", e)

Text(value='', description='Txn ID:')

Button(description='Search', style=ButtonStyle())

Output()

### Expected Output — Section 17: Colab Transaction Search and Filter GUI

**Datasets to use:**

- Final transaction dispute report DataFrame.

**How to approach:**

- Use `ipywidgets` for a simple Colab interface.
- Provide transaction ID search.
- Provide filters for priority, refund issue/status, payment mode, city, ticket status, and/or merchant category.
- Handle invalid transaction IDs and empty filters with friendly messages.
- Display only business-readable columns in the GUI output.

**Expected output format:**

The GUI should allow an operations user to:

| User Action | Expected Result |
|---|---|
| Enter valid transaction ID | Shows transaction case summary |
| Enter invalid transaction ID | Shows friendly `No transaction found` message |
| Select priority filter | Shows matching priority cases |
| Select multiple filters | Shows filtered records without modifying master dataset |

**Hint:** Keep the GUI simple. This is not a web app. It only needs to work inside Colab without editing backend variables.


In [28]:
# ============================================================
# Section 18: Report export
# Status: Broken
# ============================================================

# TODO: Export all PRD-required reports.
# BUG: output folder may not exist.
# BUG: variable names do not match final DataFrames.

# Create the output directory if it does not exist
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Use merged_df for the final transaction dispute report, as it's the most complete one so far
# The variable name final_transaction_dispute_report is not defined yet.
merged_df.to_csv(OUTPUT_PATH / "final_transaction_dispute_report.csv", index=False)

# The following reports are not yet generated by previous sections and will cause NameErrors.
# They should be created in their respective sections before attempting to export here.
# refund_pending_report.to_csv(OUTPUT_PATH / "refund_pending_report.csv", index=False)
# merchant_dispute_summary.to_csv(OUTPUT_PATH / "merchant_dispute_summary.csv", index=False)
# payment_mode_failure_summary.to_csv(OUTPUT_PATH / "payment_mode_failure_summary.csv", index=False)
# city_issue_summary.to_csv(OUTPUT_PATH / "city_issue_summary.csv", index=False)
# ai_support_prompts.to_csv(OUTPUT_PATH / "ai_support_prompts.csv", index=False)

### Expected Output — Section 18: Report Export and Report Preview

**Datasets to use:**

- Final transaction dispute report
- Refund pending report
- Merchant dispute summary
- Payment mode failure summary
- City issue summary
- P0/P1 priority cases
- AI support prompts
- Data validation report
- Debug fix log
- Transaction health summary

**How to approach:**

- Create the output folder before exporting.
- Use consistent file names from the PRD.
- Export CSV reports.
- Display a preview of each major report inside the notebook because the final submission is the notebook itself.

**Expected output format:**

Generate and preview these reports:

| Report Variable | Output File |
|---|---|
| `final_transaction_dispute_report` | `final_transaction_dispute_report.csv` |
| `refund_pending_report` | `refund_pending_report.csv` |
| `merchant_dispute_summary` | `merchant_dispute_summary.csv` |
| `payment_mode_failure_summary` | `payment_mode_failure_summary.csv` |
| `city_issue_summary` | `city_issue_summary.csv` |
| `p0_p1_priority_cases` | `p0_p1_priority_cases.csv` |
| `ai_support_prompts` | `ai_support_prompts.csv` |
| `data_validation_report` | `data_validation_report.csv` |
| `debug_fix_log_df` | `debug_fix_log.csv` |
| `transaction_health_summary` | `transaction_health_summary.csv` |

**Hint:** Even though CSV files are generated, your evaluator should be able to see report previews directly in this notebook.


In [30]:
# ============================================================
# Section 19: Final validation checks
# Status: Incomplete
# ============================================================

# TODO: Add validation checks before submission.
# Examples:
# - Final report exists and has required columns
# - Every transaction has one priority label
# - P0/P1/P2/P3 cases have priority reasons
# - Output files exist
# - Notebook runs from top to bottom after fixes

required_final_columns = [
    "transaction_id", "customer_id", "merchant_id", "amount", "payment_mode",
    "transaction_status", "refund_status", "refund_issue_tag", "dispute_priority",
    "priority_reason", "recommended_action", "ai_support_prompt"
]

# Check if merged_df exists and has the required columns
# The previous export used `merged_df` for 'final_transaction_dispute_report.csv'
missing_final_cols = [c for c in required_final_columns if c not in merged_df.columns]

if missing_final_cols:
    print(f"Warning: The following required columns are missing from merged_df: {missing_final_cols}")
else:
    print("All required final columns are present in merged_df.")

missing_final_cols

['priority_reason']

### Expected Output — Section 19: Final Validation Checks

**Datasets to use:**

- Final report DataFrames
- Exported output folder
- Debug log
- AI prompt usage log
- PRD completion mapping

**How to approach:**

- Write validation checks that prove the notebook is ready for submission.
- Check required columns, row counts, priority labels, priority reasons, recommended actions, AI prompts, output files, and debug log length.
- Show a final validation summary table with Passed/Failed status.

**Expected output format:**

Create a visible `final_validation_summary` table like:

| check_name | expected_result | actual_result | status |
|---|---|---|---|
| Final report exists | DataFrame created | Created with 5000 rows | Passed |
| Required columns present | 25+ required columns | All present | Passed |
| Valid priorities only | P0/P1/P2/P3/No Issue | Valid | Passed |
| Priority reasons present | Required for P0-P3 | Present | Passed |
| AI prompts generated | Required for P0/P1/P2 | Present | Passed |
| Debug log length | At least 10 rows | 10+ rows | Passed |
| Output files created | All required files | All present | Passed |

**Hint:** A notebook that produces reports but has no validation evidence is not evaluation-ready.


# Required In-Notebook Deliverable Workspaces

Your final submission is **one completed Colab notebook only**. Because you are not submitting separate documents, you must complete the following deliverable sections directly inside this notebook.

Each section below gives you the required format. Replace placeholder rows with your actual work and keep the outputs visible.


## Deliverable 1 — Debug Fix Log

**Purpose:** Show how you debugged and completed the resigned developer's unfinished codebase.

**Where to use it:** Update this log throughout Sections 4–19.

**Minimum requirement:** At least **10 meaningful fixes**.

**Expected output format:** Fill the table below and display it as `debug_fix_log_df`.


In [31]:
# ============================================================
# Deliverable 1 Placeholder: Debug Fix Log
# Replace sample rows with your actual debugging evidence.
# Keep this output visible in your final submitted notebook.
# ============================================================

debug_fix_log_df = pd.DataFrame([
    {
        "issue_id": "BUG-001",
        "code_section": "Section 5 - DataLoader",
        "issue_type": "Runtime / File Handling",
        "issue_description": "Example: Excel file loader was using the wrong Pandas function.",
        "root_cause": "Example: Previous developer used pd.read_csv() for refunds.xlsx.",
        "fix_summary": "Example: Replaced with pd.read_excel() and added file existence handling.",
        "tested_status": "Passed",
        "remarks": "Replace this sample row with your actual fix notes."
    },
    {
        "issue_id": "BUG-002",
        "code_section": "Section __ - ______",
        "issue_type": "Logic / Data / OOP / Export / GUI",
        "issue_description": "Write what was broken.",
        "root_cause": "Write why it happened.",
        "fix_summary": "Write what you changed.",
        "tested_status": "Passed / Failed",
        "remarks": "Add evidence or notes."
    }
])

debug_fix_log_df


,issue_id,code_section,issue_type,issue_description,root_cause,fix_summary,tested_status,remarks
0,BUG-001,Section 5 - DataLoader,Runtime / File Handling,Example: Excel file loader was using the wrong...,Example: Previous developer used pd.read_csv()...,Example: Replaced with pd.read_excel() and add...,Passed,Replace this sample row with your actual fix n...
1,BUG-002,Section __ - ______,Logic / Data / OOP / Export / GUI,Write what was broken.,Write why it happened.,Write what you changed.,Passed / Failed,Add evidence or notes.


## Deliverable 2 — AI Prompt Usage Log

**Purpose:** Since AI usage is allowed, document how you used AI responsibly.

**What to include:** Prompts used for debugging, logic design, Pandas grouping, GUI building, validation checks, or report formatting.

**Important:** Do not just paste AI answers blindly. Show how you verified or modified them.

**Expected output format:** Fill the table below and display it as `ai_prompt_usage_log_df`.


In [39]:
# ============================================================
# Deliverable 2 Placeholder: AI Prompt Usage Log
# Add the actual prompts you used while solving this notebook.
# ============================================================

ai_prompt_usage_log_df = pd.DataFrame([
    {
        "prompt_id": "AI-001",
        "section_used": "Section 9 - Refund SLA Logic",
        "prompt_goal": "Example: Understand how to calculate refund delay using pending and completed dates.",
        "prompt_used": "Example: I have refund initiated/completed dates and a fixed analysis date. How should I calculate refund delay safely in Pandas?",
        "ai_output_summary": "Example: AI suggested using completed date when available, otherwise analysis date, with pd.to_datetime and null handling.",
        "accepted_or_modified": "Modified",
        "verification_notes": "Example: I checked value counts and manually tested a few pending/completed refund records."
    },
    {
        "prompt_id": "AI-002",
        "section_used": "Section __ - ______",
        "prompt_goal": "What were you trying to solve?",
        "prompt_used": "Paste your actual prompt here.",
        "ai_output_summary": "Summarize what AI suggested.",
        "accepted_or_modified": "Accepted / Modified / Rejected",
        "verification_notes": "How did you verify the answer?"
    }
])

ai_prompt_usage_log_df


,prompt_id,section_used,prompt_goal,prompt_used,ai_output_summary,accepted_or_modified,verification_notes
0,AI-001,Section 9 - Refund SLA Logic,Example: Understand how to calculate refund de...,Example: I have refund initiated/completed dat...,Example: AI suggested using completed date whe...,Modified,Example: I checked value counts and manually t...
1,AI-002,Section __ - ______,What were you trying to solve?,Paste your actual prompt here.,Summarize what AI suggested.,Accepted / Modified / Rejected,How did you verify the answer?


## Deliverable 3 — PRD Completion Mapping

**Purpose:** Prove that your implementation matches the PRD and is not just random analysis.

**How to approach:** Map each PRD requirement to the notebook section where you implemented it.

**Expected output format:** Fill the table below and display it as `prd_completion_mapping_df`.


In [38]:
# ============================================================
# Deliverable 3 Placeholder: PRD Completion Mapping
# Map PRD requirements to your completed notebook sections.
# ============================================================

prd_completion_mapping_df = pd.DataFrame([
    {"prd_requirement": "FR-01 Multi-File Data Loader", "notebook_section": "Section 5", "implementation_summary": "Loads CSV, Excel, JSON, and TXT files with status table.", "status": "Completed", "evidence_output": "Loading summary table visible"},
    {"prd_requirement": "FR-02 Data Validation Engine", "notebook_section": "Section 6", "implementation_summary": "Required column and data quality checks implemented.", "status": "Completed / Partial / Pending", "evidence_output": "data_validation_report displayed"},
    {"prd_requirement": "FR-03 Data Cleaning and Standardization", "notebook_section": "Section 7", "implementation_summary": "Describe your cleaning logic.", "status": "Completed / Partial / Pending", "evidence_output": "Before/after value counts displayed"},
    {"prd_requirement": "FR-04 Transaction Failure Analysis", "notebook_section": "Section 8", "implementation_summary": "Describe your transaction health logic.", "status": "Completed / Partial / Pending", "evidence_output": "transaction_health_summary displayed"},
    {"prd_requirement": "FR-05 Refund Delay Analysis", "notebook_section": "Section 9", "implementation_summary": "Describe refund delay and issue tag logic.", "status": "Completed / Partial / Pending", "evidence_output": "refund_issue_tag counts displayed"},
    {"prd_requirement": "FR-06 Complaint Linking Engine", "notebook_section": "Section 10", "implementation_summary": "Describe complaint aggregation and linking.", "status": "Completed / Partial / Pending", "evidence_output": "Complaint summary displayed"},
    {"prd_requirement": "FR-07 Support Ticket Mapping", "notebook_section": "Section 11", "implementation_summary": "Describe ticket SLA mapping.", "status": "Completed / Partial / Pending", "evidence_output": "Ticket severity counts displayed"},
    {"prd_requirement": "FR-08 Duplicate Transaction Detection", "notebook_section": "Section 12", "implementation_summary": "Describe duplicate suspicion logic.", "status": "Completed / Partial / Pending", "evidence_output": "Duplicate suspected cases displayed"},
    {"prd_requirement": "FR-09 Dispute Priority Classification", "notebook_section": "Section 14", "implementation_summary": "Describe P0>P1>P2>P3>No Issue logic.", "status": "Completed / Partial / Pending", "evidence_output": "Priority counts and sample cases displayed"},
    {"prd_requirement": "FR-10 Customer Impact Score", "notebook_section": "Section 14 or separate section", "implementation_summary": "Describe score calculation.", "status": "Completed / Partial / Pending", "evidence_output": "Impact score summary displayed"},
    {"prd_requirement": "FR-14 Transaction Search GUI", "notebook_section": "Section 17", "implementation_summary": "Describe search and filter GUI.", "status": "Completed / Partial / Pending", "evidence_output": "GUI visible in notebook"},
    {"prd_requirement": "FR-17 AI-Ready Support Prompt Generator", "notebook_section": "Section 16", "implementation_summary": "Describe prompt generation for P0/P1/P2.", "status": "Completed / Partial / Pending", "evidence_output": "AI prompt examples displayed"},
    {"prd_requirement": "FR-19 Final Report Export", "notebook_section": "Section 18", "implementation_summary": "Describe exported reports.", "status": "Completed / Partial / Pending", "evidence_output": "Report previews and file list displayed"},
])

prd_completion_mapping_df


,prd_requirement,notebook_section,implementation_summary,status,evidence_output
0,FR-01 Multi-File Data Loader,Section 5,"Loads CSV, Excel, JSON, and TXT files with sta...",Completed,Loading summary table visible
1,FR-02 Data Validation Engine,Section 6,Required column and data quality checks implem...,Completed / Partial / Pending,data_validation_report displayed
2,FR-03 Data Cleaning and Standardization,Section 7,Describe your cleaning logic.,Completed / Partial / Pending,Before/after value counts displayed
3,FR-04 Transaction Failure Analysis,Section 8,Describe your transaction health logic.,Completed / Partial / Pending,transaction_health_summary displayed
4,FR-05 Refund Delay Analysis,Section 9,Describe refund delay and issue tag logic.,Completed / Partial / Pending,refund_issue_tag counts displayed
5,FR-06 Complaint Linking Engine,Section 10,Describe complaint aggregation and linking.,Completed / Partial / Pending,Complaint summary displayed
6,FR-07 Support Ticket Mapping,Section 11,Describe ticket SLA mapping.,Completed / Partial / Pending,Ticket severity counts displayed
7,FR-08 Duplicate Transaction Detection,Section 12,Describe duplicate suspicion logic.,Completed / Partial / Pending,Duplicate suspected cases displayed
8,FR-09 Dispute Priority Classification,Section 14,Describe P0>P1>P2>P3>No Issue logic.,Completed / Partial / Pending,Priority counts and sample cases displayed
9,FR-10 Customer Impact Score,Section 14 or separate section,Describe score calculation.,Completed / Partial / Pending,Impact score summary displayed


## Deliverable 4 — Assumption and Limitation Log

**Purpose:** In real engineering work, assumptions must be visible. If you make a business or technical decision not explicitly stated in the PRD, document it here.

**Expected output format:** Fill the table below and display it as `assumption_limitation_log_df`.


In [37]:
# ============================================================
# Deliverable 4 Placeholder: Assumption and Limitation Log
# ============================================================

assumption_limitation_log_df = pd.DataFrame([
    {
        "item_id": "ASM-001",
        "type": "Assumption",
        "section": "Section 9 - Refund SLA",
        "description": "Example: For pending refunds, refund delay is calculated from refund initiation date to ANALYSIS_DATE.",
        "reason": "PRD requires reproducible analysis using fixed analysis date.",
        "impact": "Ensures pending refund SLA breach detection is consistent."
    },
    {
        "item_id": "LIM-001",
        "type": "Limitation",
        "section": "Section 17 - GUI",
        "description": "Example: GUI works inside Colab only and is not a production web dashboard.",
        "reason": "Out of scope includes Flask/Streamlit/deployment.",
        "impact": "Suitable for internal notebook workflow only."
    }
])

assumption_limitation_log_df


,item_id,type,section,description,reason,impact
0,ASM-001,Assumption,Section 9 - Refund SLA,"Example: For pending refunds, refund delay is ...",PRD requires reproducible analysis using fixed...,Ensures pending refund SLA breach detection is...
1,LIM-001,Limitation,Section 17 - GUI,Example: GUI works inside Colab only and is no...,Out of scope includes Flask/Streamlit/deployment.,Suitable for internal notebook workflow only.


## Deliverable 5 — Final Report Preview Evidence

**Purpose:** Since the final submission is only this notebook, preview the final reports here.

**How to approach:** After exporting reports, display the shape and first few rows of each key report.

**Expected output format:** Run this after your final report variables are created. Add/remove report variables only if your names differ, but keep the evidence visible.


In [36]:
# ============================================================
# Deliverable 5 Placeholder: Final Report Preview Evidence
# Run after creating final report DataFrames.
# ============================================================

report_objects = {
    "final_transaction_dispute_report": globals().get("final_transaction_dispute_report"),
    "refund_pending_report": globals().get("refund_pending_report"),
    "merchant_dispute_summary": globals().get("merchant_dispute_summary"),
    "payment_mode_failure_summary": globals().get("payment_mode_failure_summary"),
    "city_issue_summary": globals().get("city_issue_summary"),
    "p0_p1_priority_cases": globals().get("p0_p1_priority_cases"),
    "ai_support_prompts": globals().get("ai_support_prompts"),
    "data_validation_report": globals().get("data_validation_report"),
    "transaction_health_summary": globals().get("transaction_health_summary"),
}

for report_name, report_df in report_objects.items():
    print("\n" + "="*80)
    print(report_name)
    if isinstance(report_df, pd.DataFrame):
        print("Shape:", report_df.shape)
        display(report_df.head())
    else:
        print("Not created yet. Create this report before final submission.")


final_transaction_dispute_report
Not created yet. Create this report before final submission.

refund_pending_report
Not created yet. Create this report before final submission.

merchant_dispute_summary
Not created yet. Create this report before final submission.

payment_mode_failure_summary
Not created yet. Create this report before final submission.

city_issue_summary
Not created yet. Create this report before final submission.

p0_p1_priority_cases
Not created yet. Create this report before final submission.

ai_support_prompts
Not created yet. Create this report before final submission.

data_validation_report
Not created yet. Create this report before final submission.

transaction_health_summary
Not created yet. Create this report before final submission.


## Deliverable 6 — Final Product Walkthrough

**Purpose:** Explain your completed product like a Python Developer handing over work to Product and Engineering.

**Expected format:** Replace the prompts below with your final explanation.


### Final Product Walkthrough — Student Response Placeholder

#### 1. Business Problem
Write 4–6 lines explaining the transaction dispute problem and why the operations team needs this tool.

#### 2. Product Flow
Explain the final notebook flow:

1. Dataset upload and extraction
2. Multi-file loading
3. Validation
4. Cleaning
5. Refund, complaint, ticket, duplicate analysis
6. Priority classification
7. Recommended action and AI prompt generation
8. GUI search/filter
9. Report export and validation

#### 3. Major Bugs Fixed
Summarize the most important 5–7 bugs you fixed from the previous developer's codebase.

#### 4. Most Important Business Rules Implemented
Explain refund SLA, complaint severity, ticket severity, duplicate suspicion, priority hierarchy, and recommended action logic.

#### 5. Final Reports Generated
List the reports generated and what each report is used for.

#### 6. How an Operations User Will Use This Colab Product
Explain how a support executive or operations analyst can use the search/filter/report sections.

#### 7. Assumptions and Limitations
Summarize the most important assumptions and limitations from your assumption log.

#### 8. Final Readiness Statement
Write whether the product is ready for internal simulated handoff and what evidence proves it.


## Deliverable 7 — Final Self-Check Before Submission

Use this section to prove that your single notebook submission is complete.


In [34]:
# ============================================================
# Deliverable 7 Placeholder: Final Self-Check
# Update actual_result/status after your implementation is complete.
# ============================================================

final_self_check_df = pd.DataFrame([
    {"check_item": "Notebook runs from top to bottom", "expected_result": "No unresolved errors", "actual_result": "To be filled", "status": "Pending"},
    {"check_item": "All datasets loaded", "expected_result": "7 files loaded", "actual_result": "To be filled", "status": "Pending"},
    {"check_item": "Data validation report visible", "expected_result": "data_validation_report displayed", "actual_result": "To be filled", "status": "Pending"},
    {"check_item": "Debug fix log visible", "expected_result": "10+ meaningful rows", "actual_result": "To be filled", "status": "Pending"},
    {"check_item": "AI prompt usage log visible", "expected_result": "AI usage documented", "actual_result": "To be filled", "status": "Pending"},
    {"check_item": "PRD mapping visible", "expected_result": "Major FRs mapped", "actual_result": "To be filled", "status": "Pending"},
    {"check_item": "Final reports previewed", "expected_result": "Report shapes and previews visible", "actual_result": "To be filled", "status": "Pending"},
    {"check_item": "GUI works", "expected_result": "Search and filters usable in Colab", "actual_result": "To be filled", "status": "Pending"},
    {"check_item": "Final walkthrough completed", "expected_result": "Explanation written in notebook", "actual_result": "To be filled", "status": "Pending"},
])

final_self_check_df


,check_item,expected_result,actual_result,status
0,Notebook runs from top to bottom,No unresolved errors,To be filled,Pending
1,All datasets loaded,7 files loaded,To be filled,Pending
2,Data validation report visible,data_validation_report displayed,To be filled,Pending
3,Debug fix log visible,10+ meaningful rows,To be filled,Pending
4,AI prompt usage log visible,AI usage documented,To be filled,Pending
5,PRD mapping visible,Major FRs mapped,To be filled,Pending
6,Final reports previewed,Report shapes and previews visible,To be filled,Pending
7,GUI works,Search and filters usable in Colab,To be filled,Pending
8,Final walkthrough completed,Explanation written in notebook,To be filled,Pending


# Evaluation Rubric — 100 Marks

|   No | Criteria                                                |   Marks | Expectation                                                                                                                                                |
|-----:|:--------------------------------------------------------|--------:|:-----------------------------------------------------------------------------------------------------------------------------------------------------------|
|    1 | Colab setup, file upload flow, and multi-file loading   |       8 | Dataset ZIP upload/extraction works in Colab; all CSV/XLSX/JSON/TXT files load with clear status messages.                                                 |
|    2 | Data validation and defensive error handling            |      10 | Required columns, duplicate keys, null critical fields, orphan records, invalid files, and optional file warnings are handled without notebook crashes.    |
|    3 | Data cleaning and standardization                       |      10 | Statuses, payment modes, dates, amounts, text fields, missing values, duplicate transactions, and inconsistent categories are cleaned correctly.           |
|    4 | Refund delay and SLA business logic                     |      10 | Refund pending, SLA breach, completed on time, partial refund, failed refund, amount mismatch, and missing refund records are classified accurately.       |
|    5 | Complaint and support ticket linking                    |      10 | Complaints/tickets are merged correctly; repeated complaints, unresolved complaints, escalations, ticket delays, and slow resolution cases are identified. |
|    6 | Duplicate detection and dispute priority classification |      12 | Possible duplicate transactions are flagged; P0/P1/P2/P3/No Issue hierarchy is implemented with clear priority reasons.                                    |
|    7 | Customer impact score and recommended action generation |       8 | Impact scores are capped at 100, impact levels are assigned, and every disputed case gets a business-readable recommended action.                          |
|    8 | Pandas analysis summaries and final report exports      |      10 | Merchant, payment mode, city, refund pending, P0/P1, AI prompt, validation, debug, and health summary reports are generated correctly.                     |
|    9 | Colab GUI, transaction search, filters, and usability   |      12 | Search and filter interface works inside Colab with valid/invalid IDs and readable non-technical outputs.                                                  |
|   10 | Presentation & Explanation                              |      10 | Student clearly explains business context, codebase issues, fixes made, PRD mapping, assumptions, limitations, and final outputs.                          |

# Final Submission Checklist

Before submission, make sure your **single completed Colab notebook** contains:

- Fully fixed and runnable code
- All outputs visible after running the notebook
- Data loading, validation, cleaning, analysis, GUI, and export sections working
- Debug fix log completed in the notebook deliverable workspace
- AI prompt usage log completed in the notebook deliverable workspace
- PRD completion mapping completed in the notebook deliverable workspace
- Assumption and limitation log completed in the notebook deliverable workspace
- Final report previews visible inside the notebook
- Final validation summary visible inside the notebook
- Final product walkthrough completed inside the notebook
- Presentation/explanation notes completed inside the notebook

## Required submission format

Submit only the completed `.ipynb` file.

Do not submit CSV outputs, screenshots, PDFs, PPTs, or separate documents unless your instructor explicitly asks for them later.

## Important

A notebook with working code but missing logs, mapping, explanation, and visible outputs is **not complete** for this capstone.
